In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("Pokemon.csv")

# Cleans column names (removes unneeded white spaces in column names)
df.columns = [c.strip() for c in df.columns]

# Fills missing secondary types
df["Type2"] = df["Type2"].fillna("None")

print(df.head())

   Number        Name  Types  Type1   Type2  Evolutions  Legendary
0       1   Bulbasaur      2  grass  poison           2          0
1       2     Ivysaur      2  grass  poison           2          0
2       3    Venusaur      2  grass  poison           2          0
3       4  Charmander      1   fire    None           2          0
4       5  Charmeleon      1   fire    None           2          0


We imported our needed libraries and used the read function in pandas to read the data in the Pokemon.csv file.
Printed the "head" part of our data which displays the first 5 rows and all of the columns to check if our dataset was imported properly.

In [2]:
features = ["Types", "Type1", "Type2", "Evolutions", "Legendary"]

X = df[features].to_dict(orient="records")
y = df["Name"].tolist()

Machine Learning always seperates x as inputs and y as outputs. We first put all the features in a list called 'x' and then we translated it into a dict.    Made a list named 'y' and put all the outputs (Pokemon names into it).

In [3]:
class Node:
    def __init__(self, feature=None, value=None, children=None, prediction=None):
        self.feature = feature
        self.value = value
        self.children = children or {}
        self.prediction = prediction

Made a class named node for defining what information every node stores. 1) The feature variable stores the feature (type, legendary, etc), 2) The value variable stores the value of the feature (if the feature was legendary, then this stores if the pokemon is legendary or not), 3) The children variable stores the children of the node in a dictionary, 4) The prediction variable stores the final prediction of the Pokemon (the leaf node).

In [4]:
from collections import Counter 

def majority_class(labels):
    return Counter(labels).most_common(1)[0][0]

A majority class function, if we have multiple Pokemon at the same leaf node then this helps it select the output Pokemon depending on which appears majorly.

In [5]:
def entropy(labels):

    counts = Counter(labels)
    total = len(labels)

    ent = 0

    for c in counts.values():
        p = c / total
        ent -= p * np.log2(p)

    return ent

Made a function called entropy which finds the entropy (randomness) of each column and based on this, the model will divide the nodes of the tree.
Used a loop to print the entropy of each column.

In [6]:
def information_gain(X, y, feature):

    parent_entropy = entropy(y)
    total = len(y)

    values = set(row[feature] for row in X)

    weighted_entropy = 0

    for v in values:

        subset_y = [
            y[i] for i in range(len(X))
            if X[i][feature] == v
        ]

        weight = len(subset_y) / total

        weighted_entropy += weight * entropy(subset_y)

    return parent_entropy - weighted_entropy

This calculates Information Gain which tells us how much each feature can reduce entropy and depending on this, the tree chooses the base and decision features and splits.

In [7]:
def best_feature(X, y, features):

    best_gain = -1
    best_feat = None

    for f in features:

        gain = information_gain(X, y, f)

        if gain > best_gain:
            best_gain = gain
            best_feat = f

    return best_feat

Calculates Information Gain for each feature and then finds the best feature to base the decision tree upon.

In [8]:
def build_tree(X, y, features):

    if len(set(y)) == 1:
        return Node(prediction=y[0])

    if not features:
        return Node(prediction=majority_class(y))

    best = best_feature(X, y, features)

    node = Node(feature=best)

    values = set(row[best] for row in X)

    for v in values:

        subset_X = []
        subset_y = []

        for i in range(len(X)):
            if X[i][best] == v:
                subset_X.append(X[i])
                subset_y.append(y[i])

        if not subset_X:
            node.children[v] = Node(prediction=majority_class(y))
        else:
            remaining = [f for f in features if f != best]

            node.children[v] = build_tree(subset_X, subset_y, remaining)

    return node

Function to build the tree based using the functions written above.
First if condition is the first stopping condition which stops when the node is pure (only 1 possibility).
Second if condition is the second stopping condition which stops when there are no more features available.
Looping through the values and creating subsets for the branch.
If the feature is same as value, then it selects those rows and stores them in the subset x.
Recursively build the decision tree from this node.


In [9]:
def predict(tree, sample):

    current_node = tree

    while current_node.prediction is None:

        feature = current_node.feature
        value = sample[feature]

        if value not in current_node.children:
            return None

        current_node = current_node.children[value]

    return current_node.prediction

Predicts the output using some questions about the Pokemon.

In [16]:
tree = build_tree(X, y, features)

feature_prompts = {
    "Types": "How many types?",
    "Type1": "Primary type",
    "Type2": "Secondary type (use None if no secondary type)",
    "Evolutions": "Total evolutions in its line (e.g., Charizard = 2)",
    "Legendary": "Legendary (0 = no, 1 = yes)",
}

feature_choices = {
    f: sorted(df[f].dropna().unique().tolist())
    for f in features
}

def prompt_choice(feature, choices):
    if all(isinstance(c, (int, np.integer)) for c in choices):
        choices_str = ", ".join(str(c) for c in choices)
        while True:
            raw = input(f"{feature} ({choices_str}): ").strip()
            try:
                value = int(raw)
            except ValueError:
                print("Please enter a number from the choices.")
                continue
            if value in choices:
                return value
            print("Please choose one of the listed values.")
    else:
        choices_str = ", ".join(str(c) for c in choices)
        normalized = {str(c).strip().lower(): c for c in choices}
        while True:
            raw = input(f"{feature} ({choices_str}): ").strip().lower()
            if raw in normalized:
                return normalized[raw]
            print("Please choose one of the listed values.")

def ask_features():
    answers = {}
    for f in features:
        prompt = feature_prompts.get(f, f)
        answers[f] = prompt_choice(prompt, feature_choices[f])
    return answers

# Interactive guessing
user_sample = ask_features()
prediction = predict(tree, user_sample)

print("Your answers:", user_sample)
if prediction is None:
    print("I could not find a matching Pokemon for those answers.")
else:
    print("Predicted Pokemon:", prediction)

mask = pd.Series(True, index=df.index)
for f in features:
    mask &= df[f] == user_sample[f]

matches = df.loc[mask, "Name"].tolist()
if matches:
    if prediction in matches:
        matches = [prediction] + [n for n in matches if n != prediction]
    if len(matches) > 1:
        print("Multiple Pokemon match these features.")
    confirmed = None
    for name in matches:
        confirm = input(f"Is your Pokemon {name}? (y/n): ").strip().lower()
        if confirm in ("y", "yes"):
            confirmed = name
            break
    if confirmed:
        print(f"Great! Confirmed: {confirmed}.")
    else:
        print("Thanks! None of the matches were confirmed.")
else:
    print("No Pokemon match those exact features.")


Your answers: {'Types': 1, 'Type1': 'fire', 'Type2': 'None', 'Evolutions': 1, 'Legendary': 0}
Predicted Pokemon: Vulpix
Multiple Pokemon match these features.
Thanks! None of the matches were confirmed.
